# 05 · Temporal kNN on the d8 residual — relevance / analog probes

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first.** A model-free probe of the *same* d8@cs0.5 leftover the MoE/EBM model: predict the
h16-19 residual from **regime-similar past close bars** (self-attention ≈ learned kNN; here a fixed-metric
causal kNN). Two leakage-guarded variants (`knn_d8.sbatch`), each with a **shuffle placebo** (shuffle the
neighbour residuals → must give ~0; a gain there = look-ahead artifact).

In [ ]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))
print("setup ok")

---
## 1 · The two variants (folded)

- **`knn_analog`** — causal kNN analog: weighted **mean** of the K nearest past close analogs, strict
  embargo ≥ HAR max lag (3125 bars) so neighbour residuals are fully realised.
- **`knn_local`** — large-K **local-linear ridge** relevance regression (the "proper" variant; naive
  K-mean over-adds variance and hurts). Prior standalone: **−0.00018** on the close leftover, shuffle-clean,
  but **dominated by the EBM regime** (−0.00079). Re-run here on the linbest base for an apples-to-apples row.

In [ ]:
from inspect import getsource  # the leakage guards live in the embargo + shuffle-placebo loop
import knn_analog, knn_local
display(show_one(knn_analog.main))

---
## 2 · Verify — the kNN runs (backfills from the knn_d8 log)

`knn_d8.sbatch` runs both on the linbest `fa_d8c5` (d8@cs0.5) leftover. Parse its log into
`results/moe_ladder/knn_d8.csv` (variant, fset, qlike_full, d_full, qlike_h16_19, d_h16_19, shuffle).

In [ ]:
p = REPO / "results" / "moe_ladder" / "knn_d8.csv"
if p.exists() and p.stat().st_size > 0:
    knn = pd.read_csv(p); display(knn)
    shuf = knn[knn.fset.astype(str).str.contains("SHUFFLE", case=False, na=False)]
    if len(shuf):
        assert (shuf.d_full.abs() < 5e-4).all(), "SHUFFLE placebo not ~0 — look-ahead artifact!"
        print("PASS - shuffle placebo ~0 (leakage-clean).")
else:
    print("PENDING - knn_d8 job not yet parsed. Numbers in logs/knn_d8_<jid>.out "
          "(lines: 'Hero A: full=.. h16-19=..', 'K=.. full .. h16-19 ..', 'SHUFFLE .. [should be ~0]').")

## 3 · Interpret

The expected story (and why it matters even if it loses):

- **The proper local-linear kNN works** (placebo-clean) but is **dominated by the EBM regime** — kNN is a
  *validated method, not a deployable lever here*. The naive K-mean **hurts** (variance), which is itself
  a finding: the close leftover is low-dimensional and noise-dominated, so adding unstructured local
  variance is harmful — only *structured* local fitting (ridge) extracts the thin signal.
- **kNN vs the MoE gate (ch. 04):** the MoE's soft gate is a *learned-metric* generalization of fixed kNN
  (the gate learns the regime the kNN's hand-metric assumes). If neither beats the EBM, the state framing
  is earned for real and the lever is **information (the auction cross), not more local/relevance modelling**.
- The model class that subsumes both (sequence-attention) is **provably empty here** (log-sig path-lever
  death, ch. 03) — so the kNN is the right, data-efficient, *legible* probe, not a stepping stone to a
  sequence model.